In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    first_half = x[..., :half]
    second_half = x[..., half:]
    # print(first_half)
    # print(second_half.shape)
    new_first_half = first_half * cos - second_half * sin
    new_second_half = first_half * sin + second_half * cos
    out = torch.cat([new_first_half, new_second_half], dim = -1)
    return out 

In [3]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.d_head = d_head
        self.W_q = nn.Linear(d_model, d_head, bias = False)
        self.W_k = nn.Linear(d_model, d_head, bias = False)
        self.W_v = nn.Linear(d_model, d_head, bias = False)

    def forward(self, x, cos, sin):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = apply_rope(Q, cos, sin)
        K = apply_rope(K, cos, sin)
        scores = (1/math.sqrt(self.d_head))*(Q @ K.T)

        seq_len = x.shape[0]
        mask = torch.ones((seq_len, seq_len))
        causal_mask = torch.triu(mask, diagonal= 1)
        scores = scores.masked_fill(causal_mask.bool(), float('-inf'))

        attn_weights = torch.softmax(scores, dim = 1)
        # print(attn_weights.shape)
        # print(attn_weights)
        output = attn_weights @ V

        return output

In [4]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads):
        super().__init__()
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_q_heads
        self.group_size = num_q_heads // num_kv_heads

        self.W_q = nn.Linear(d_model, num_q_heads * self.head_dim, bias = False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.head_dim, bias = False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.head_dim, bias = False)
        self.W_o = nn.Linear(num_q_heads * self.head_dim, d_model, bias = False)

        i = torch.arange(0, self.head_dim, 2)
        self.freqs = 1.0/(10000 ** (i / self.head_dim))

    def forward(self, x):
        seq_len, d_model = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(seq_len, self.num_q_heads, self.head_dim).transpose(0, 1)
        K = K.view(seq_len, self.num_kv_heads, self.head_dim).transpose(0, 1)
        V = V.view(seq_len, self.num_kv_heads, self.head_dim).transpose(0, 1)

        K = torch.repeat_interleave(K, self.group_size, dim = 0)
        V = torch.repeat_interleave(V, self.group_size, dim = 0)

        positions = torch.arange(seq_len)
        angles = positions[:, None] * self.freqs[None,:]
        cos = torch.cos(angles)
        sin = torch.sin(angles)

        Q = apply_rope(Q, cos, sin)
        K = apply_rope(K, cos, sin)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal= 1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim = -1)
        output = attn_weights @ V
        output = output.transpose(0, 1).reshape(seq_len, self.num_q_heads * self.head_dim)
        output = self.W_o(output)
        return output

In [5]:
torch.manual_seed(0)
d_model, num_q_heads, num_kv_heads, seq_len = 2048, 32, 4, 5

gqa = GroupedQueryAttention(d_model, num_q_heads, num_kv_heads)
x = torch.rand(seq_len, d_model)
output = gqa(x)
print(output.shape)  # should be (5, 2048)

torch.Size([5, 2048])


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        d_head = d_model // num_heads
        self.heads = nn.ModuleList([
            SingleHeadAttention(d_model, d_head) for _ in range(num_heads)
        ])
        self.W_o = nn.Linear(d_model, d_model, bias = False)
        i = torch.arange(0, d_head, 2)  # indices 0, 2
        self.freqs = 1.0 / (10000 ** (i / d_head))

    def forward(self, x):
        positions = torch.arange(x.shape[0])
        angles = positions[:,None] * self.freqs[None, :]
        cos = torch.cos(angles)
        sin = torch.sin(angles)
        head_outputs = [head(x, cos, sin) for head in self.heads]
        output = torch.cat(head_outputs, dim = 1)
        output = self.W_o(output)
        return output

In [7]:
class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias = False)
        self.up_proj = nn.Linear(d_model, d_ff, bias = False)
        self.down_proj = nn.Linear(d_ff, d_model, bias = False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        fused = F.silu(gate) * up
        output = self.down_proj(fused)
        return output

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = (x / torch.sqrt(torch.mean(x**2, dim = -1, keepdim=True) + self.eps))
        output = rms * self.weight
        return output


In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, num_q_heads, num_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.mlp = MLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [10]:
class TinyLlamaModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_q_heads, num_kv_heads, d_ff, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_q_heads, num_kv_heads, d_ff) for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        x = self.embed(token_ids)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

In [11]:
torch.manual_seed(0)
vocab_size, d_model, num_q_heads, num_kv_heads, d_ff, num_layers = 32000, 2048, 32, 4, 5632, 22

model = TinyLlamaModel(vocab_size, d_model, num_q_heads, num_kv_heads, d_ff, num_layers)
token_ids = torch.randint(0, vocab_size, (5,))
logits = model(token_ids)
print(logits.shape)  # should be (5, 32000)

torch.Size([5, 32000])


In [ ]:
# torch.manual_seed(0)
# vocab_size, d_model, num_heads, d_ff, num_layers = 1000, 8, 2, 16, 4

# model = TinyLlamaModel(vocab_size, d_model, num_heads, d_ff, num_layers)
# token_ids = torch.randint(0, vocab_size, (5,))  # 5 random token IDs

# logits = model(token_ids)
# print(logits.shape)  # should be (seq_len, vocab_size) = (5, 1000)

torch.Size([5, 1000])


In [10]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
hf_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype = torch.float32)
tokenizer = AutoTokenizer.from_pretrained(model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:
for name, param in hf_model.state_dict().items():
    print(name, tuple(param.shape))

model.embed_tokens.weight (32000, 2048)
model.layers.0.self_attn.q_proj.weight (2048, 2048)
model.layers.0.self_attn.k_proj.weight (256, 2048)
model.layers.0.self_attn.v_proj.weight (256, 2048)
model.layers.0.self_attn.o_proj.weight (2048, 2048)
model.layers.0.mlp.gate_proj.weight (5632, 2048)
model.layers.0.mlp.up_proj.weight (5632, 2048)
model.layers.0.mlp.down_proj.weight (2048, 5632)
model.layers.0.input_layernorm.weight (2048,)
model.layers.0.post_attention_layernorm.weight (2048,)
model.layers.1.self_attn.q_proj.weight (2048, 2048)
model.layers.1.self_attn.k_proj.weight (256, 2048)
model.layers.1.self_attn.v_proj.weight (256, 2048)
model.layers.1.self_attn.o_proj.weight (2048, 2048)
model.layers.1.mlp.gate_proj.weight (5632, 2048)
model.layers.1.mlp.up_proj.weight (5632, 2048)
model.layers.1.mlp.down_proj.weight (2048, 5632)
model.layers.1.input_layernorm.weight (2048,)
model.layers.1.post_attention_layernorm.weight (2048,)
model.layers.2.self_attn.q_proj.weight (2048, 2048)
mode

In [12]:
config = hf_model.config
print("num_attention_heads (query heads):", config.num_attention_heads)
print("num_key_value_heads:", config.num_key_value_heads)
print("hidden_size (d_model):", config.hidden_size)
print("head_dim:", config.hidden_size // config.num_attention_heads)
print("num_hidden_layers:", config.num_hidden_layers)
print("intermediate_size (d_ff):", config.intermediate_size)
print("rope_theta:", getattr(config, "rope_theta", "not set, defaults to 10000"))

num_attention_heads (query heads): 32
num_key_value_heads: 4
hidden_size (d_model): 2048
head_dim: 64
num_hidden_layers: 22
intermediate_size (d_ff): 5632
rope_theta: not set, defaults to 10000


In [13]:
seq_len, d_model = 5, 2048
num_q_heads, head_dim = 32, 64

Q = torch.rand(seq_len, d_model)

Q_heads = Q.view(seq_len, num_q_heads, head_dim)
Q_heads = Q_heads.transpose(0, 1)
print(Q_heads.shape)

torch.Size([32, 5, 64])


In [22]:
num_kv_heads = 4
group_size = num_q_heads // num_kv_heads

K = torch.rand(seq_len, num_kv_heads * head_dim)

K_heads = K.view(seq_len, num_kv_heads, head_dim).transpose(0,1)
K_heads_expanded = torch.repeat_interleave(K_heads, group_size, dim = 0)
print(K_heads_expanded.shape)

torch.Size([32, 5, 64])


In [23]:
V = torch.rand(seq_len, num_kv_heads * head_dim)
V_heads = V.view(seq_len, num_kv_heads, head_dim).transpose(0, 1)
V_heads_expanded = torch.repeat_interleave(V_heads, group_size, dim=0)
print(V_heads_expanded.shape)  # expect (32, 5, 64)

torch.Size([32, 5, 64])


In [25]:
scores = Q_heads @ K_heads_expanded.transpose(-2, -1) / math.sqrt(head_dim)  # (32, 5, 5)
print(scores.shape)

torch.Size([32, 5, 5])


In [27]:
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()  # (5, 5)
scores = scores.masked_fill(mask, float('-inf'))  # apply mask to all 32 heads at once

attn_weights = torch.softmax(scores, dim=-1)  # which dimension is "keys" here?

output = attn_weights @ V_heads_expanded  # (32, 5, 5) @ (32, 5, 64) → ?
print(output.shape)

torch.Size([32, 5, 64])


In [28]:
print(attn_weights[0].sum(dim=-1))  # head 0's attention weights — each row should sum to 1
print(attn_weights[0])  # should show the same lower-triangular causal pattern as your very first single-head test

tensor([1., 1., 1., 1., 1.])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5125, 0.4875, 0.0000, 0.0000, 0.0000],
        [0.2967, 0.3185, 0.3848, 0.0000, 0.0000],
        [0.2244, 0.2493, 0.3084, 0.2179, 0.0000],
        [0.1712, 0.2053, 0.2521, 0.1847, 0.1868]])


In [30]:
d_model = num_q_heads * head_dim
output = output.transpose(0, 1)  # (5, 32, 64) — heads back to second position
output = output.reshape(seq_len, d_model)  # (5, 2048) — merge heads back into one vector
print(output.shape)

torch.Size([5, 2048])


In [14]:
torch.manual_seed(0)
d_model, num_heads, seq_len = 8, 2, 5

mha = MultiHeadAttention(d_model, num_heads)
x = torch.rand(seq_len, d_model)

output = mha(x)
print(output.shape)  # should be (5, 8)

torch.Size([5, 8])


In [15]:
d_head = 8  # small example, 2 pairs
i = torch.arange(0, d_head, 2)  # indices 0, 2
print(i)
freqs = 1.0 / (10000 ** (i / d_head))
print(freqs)

tensor([0, 2, 4, 6])
tensor([1.0000, 0.1000, 0.0100, 0.0010])


In [16]:
seq_len = 5
positions = torch.arange(seq_len)  # [0, 1, 2, 3, 4]

# angle for position m, frequency i = m * freq_i
# we want a (seq_len, num_pairs) table of angles
print(positions[:,None])
print(freqs[None,:])
angles = positions[:,None] * freqs[None,:]  # hint: positions[:, None] * freqs[None, :]
print(angles)

cos = torch.cos(angles)
sin = torch.sin(angles)
print(cos.shape, sin.shape)  # should be (seq_len, num_pairs) = (5, 2)

tensor([[0],
        [1],
        [2],
        [3],
        [4]])
tensor([[1.0000, 0.1000, 0.0100, 0.0010]])
tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03],
        [4.0000e+00, 4.0000e-01, 4.0000e-02, 4.0000e-03]])
torch.Size([5, 4]) torch.Size([5, 4])


In [17]:
x = torch.rand([5,4])
print(x)
# apply_rope(x, cos, sin)

tensor([[0.8458, 0.1278, 0.7048, 0.3319],
        [0.2588, 0.5898, 0.2403, 0.6152],
        [0.5982, 0.1288, 0.5832, 0.7130],
        [0.6979, 0.4371, 0.0901, 0.4229],
        [0.6737, 0.3176, 0.6898, 0.8330]])
